# 02 — Data Cleaning and Metric Computation

This notebook takes the raw shot log from `data/raw/shots_2025_26.csv` and prepares it for all downstream analysis. It is the single data pipeline step between the raw API pull and every analysis notebook that follows.

What this notebook does, in order:

1. Load the raw 2025-26 NBA shot log
2. Clean and convert shot coordinates from tenths of feet to feet with the basket at the origin
3. Filter out rows with missing or physically impossible coordinates
4. Assign each shot to one of 11 named court zones using geometry from `src/metrics.py`, with SHOT_TYPE as the final arbiter at zone boundaries — no shot can appear in both a 2-point and a 3-point zone
5. Apply the 300-shot minimum filter at the player level — players below this threshold are excluded from the entire dataset
6. Compute Points Per Shot (PPS) and shot frequency per player per zone with no zone-level minimum filter — every zone with at least 1 attempt is included
7. Compute league average PPS per zone across all qualifying players
8. Save the enriched shot-level data to `data/processed/shots_cleaned.csv`
9. Save the zone-level summary table to `data/processed/zone_stats_2025_26.csv`

The only efficiency metric used in this notebook and throughout the project is PPS. A made 2-pointer is worth 2 points, a made 3-pointer is worth 3 points, and any miss is worth 0 points. eFG% does not appear anywhere.

## Section 1.1 — Imports, Constants, and Paths

All imports for this notebook live here in a single cell. Nothing is imported anywhere else. Running this cell first is the only setup step required before any other cell.

Constants defined here:

`SEASON` — the NBA season this analysis covers. Changing this one value updates all file paths automatically.

`MIN_PLAYER_FGA` — the minimum number of total shot attempts required for a player to be included in the analysis. Set to 300 per the project definition.

`ZONE_ORDER` — the canonical ordering of all 11 court zones used in every table and chart throughout the project. Defined once here and referenced everywhere else.

`THREE_PT_ZONES` — the set of zone names that are 3-point territory. Used in the SHOT_TYPE arbiter step to resolve coordinate/classification conflicts at zone boundaries.

File paths are all built programmatically from `PROJECT_ROOT` so they work correctly regardless of where on your machine the repo lives.

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add the project root to Python's module search path so we can import from src/
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import assign_zone   # geometry-based zone classifier — never redefine this here

# ── Season and player pool ─────────────────────────────────────────────────────
SEASON         = "2025-26"   # change this to re-run for a different season
MIN_PLAYER_FGA = 300         # players below this total shot count are excluded

# ── Zone ordering — defined once, used throughout ─────────────────────────────
ZONE_ORDER = [
    "Restricted Area",
    "Paint (Non-RA)",
    "Mid-Range Left",
    "Mid-Range Center",
    "Mid-Range Right",
    "Left Corner 3",
    "Right Corner 3",
    "Above Break 3 Left",
    "Above Break 3 Center",
    "Above Break 3 Right",
    "Backcourt",
]

# ── 3-point zones — used by the SHOT_TYPE arbiter in Section 2.1 ──────────────
THREE_PT_ZONES = frozenset({
    "Left Corner 3",
    "Right Corner 3",
    "Above Break 3 Left",
    "Above Break 3 Center",
    "Above Break 3 Right",
})

# ── File paths — all derived from PROJECT_ROOT ────────────────────────────────
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

_season_slug   = SEASON.replace("-", "_")           # "2025-26" → "2025_26"
RAW_FILE       = RAW_DIR       / f"shots_{_season_slug}.csv"
SHOTS_OUT      = PROCESSED_DIR / "shots_cleaned.csv"
ZONE_STATS_OUT = PROCESSED_DIR / f"zone_stats_{_season_slug}.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ZONE_STATS_OUT = PROCESSED_DIR / f"zone_stats_{_season_slug}.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Confirmation print ────────────────────────────────────────────────────────
print(f"Season          : {SEASON}")
print(f"Min player FGA  : {MIN_PLAYER_FGA}")
print(f"Raw input       : {RAW_FILE}")
print(f"Shots output    : {SHOTS_OUT}")
print(f"Zone stats out  : {ZONE_STATS_OUT}")
print(f"assign_zone     : imported from src.metrics")
print(f"Zones defined   : {len(ZONE_ORDER)}")

Season          : 2025-26
Min player FGA  : 300
Raw input       : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/raw/shots_2025_26.csv
Shots output    : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/shots_cleaned.csv
Zone stats out  : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/zone_stats_2025_26.csv
assign_zone     : imported from src.metrics
Zones defined   : 11


## Section 1.2 — Load the Raw Data

We read the CSV saved by Notebook 01 into a DataFrame called `raw` and leave it completely untouched. Every cleaning step that follows works on a copy so we can always compare the cleaned result against the original if something looks wrong.

Three checks tell us what we are starting with:

`.shape` confirms the row count (shot attempts) and column count. Based on the Notebook 01 pull we expect roughly 219,000 rows.

`.head(3)` gives a visual confirmation that column names and values look reasonable before any code runs on them.

`.info()` shows every column's data type and its non-null count. A column with fewer non-null entries than total rows has missing values that need handling. We want to see zero nulls across all columns at this stage.

In [2]:
raw = pd.read_csv(RAW_FILE)

print(f"Shape: {raw.shape[0]:,} rows  x  {raw.shape[1]} columns\n")

print("First 3 rows:")
display(raw.head(3))

print("\nColumn types and null counts:")
raw.info()

Shape: 219,160 rows  x  25 columns

First 3 rows:


,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,NAME
0,Shot Chart Detail,22500038,224,1630173,Precious Achiuwa,1610612758,Sacramento Kings,2,7,54,...,24+ ft.,27,-50,274,1,0,20251107,SAC,OKC,Precious Achiuwa
1,Shot Chart Detail,22500038,259,1630173,Precious Achiuwa,1610612758,Sacramento Kings,2,5,27,...,Less Than 8 ft.,4,-40,8,1,1,20251107,SAC,OKC,Precious Achiuwa
2,Shot Chart Detail,22500038,659,1630173,Precious Achiuwa,1610612758,Sacramento Kings,4,3,48,...,Less Than 8 ft.,4,17,40,1,0,20251107,SAC,OKC,Precious Achiuwa



Column types and null counts:
<class 'pandas.DataFrame'>
RangeIndex: 219160 entries, 0 to 219159
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   GRID_TYPE            219160 non-null  str  
 1   GAME_ID              219160 non-null  int64
 2   GAME_EVENT_ID        219160 non-null  int64
 3   PLAYER_ID            219160 non-null  int64
 4   PLAYER_NAME          219160 non-null  str  
 5   TEAM_ID              219160 non-null  int64
 6   TEAM_NAME            219160 non-null  str  
 7   PERIOD               219160 non-null  int64
 8   MINUTES_REMAINING    219160 non-null  int64
 9   SECONDS_REMAINING    219160 non-null  int64
 10  EVENT_TYPE           219160 non-null  str  
 11  ACTION_TYPE          219160 non-null  str  
 12  SHOT_TYPE            219160 non-null  str  
 13  SHOT_ZONE_BASIC      219160 non-null  str  
 14  SHOT_ZONE_AREA       219160 non-null  str  
 15  SHOT_ZONE_RANGE      219160 non

## Section 1.3 — Understanding the NBA API Coordinate System

Before cleaning the coordinates, we need to understand exactly what the raw values mean. The NBA API encodes shot locations in a non-obvious way and getting this wrong corrupts every zone assignment that follows.

**Unit:** LOC_X and LOC_Y are in tenths of a foot, not feet. A raw value of 237 means 23.7 feet, not

**Origin:** The basket sits at (0, 0). All distances are measured from the center of the rim.

**LOC_X — horizontal position:**
Negative values are the left side of the court, positive are the right side. The sidelines sit at roughly ±250 in raw units (±25 feet). A left corner three will show LOC_X near −220 (−22 feet).

**LOC_Y — depth from the basket toward half court:**
Zero is directly at the basket. Positive values go toward half court. The baseline behind the basket sits at around −47 in raw units (−4.75 feet). A shot at the top of the three-point arc will show LOC_Y near 237 (23.7 feet).

The code cell below prints descriptive statistics for both columns so we can see the actual range of values in this season's data and spot anything unexpected before proceeding. We also save the raw min and max values here so we can print a before/after comparison after coordinate cleaning is complete.

In [3]:
# Confirm no nulls — we expect zero from Section 1.2 but verify explicitly
# because the cleaning step in Section 1.4 depends on this being true
null_x = raw["LOC_X"].isna().sum()
null_y = raw["LOC_Y"].isna().sum()
print(f"Null values — LOC_X: {null_x}  |  LOC_Y: {null_y}\n")

# Descriptive statistics for both coordinate columns in raw tenths-of-foot units
print("Raw coordinate statistics (tenths of a foot):\n")
print(raw[["LOC_X", "LOC_Y"]].describe().round(1).to_string())

# Save raw ranges now — used for the before/after summary in Section 1.6
RAW_X_RANGE = (int(raw["LOC_X"].min()), int(raw["LOC_X"].max()))
RAW_Y_RANGE = (int(raw["LOC_Y"].min()), int(raw["LOC_Y"].max()))

print(f"\nLOC_X range : {RAW_X_RANGE[0]} → {RAW_X_RANGE[1]}  "
      f"(expected roughly −250 → 250)")
print(f"LOC_Y range : {RAW_Y_RANGE[0]} → {RAW_Y_RANGE[1]}  "
      f"(expected roughly −50 → 750)")

Null values — LOC_X: 0  |  LOC_Y: 0

Raw coordinate statistics (tenths of a foot):

          LOC_X     LOC_Y
count  219160.0  219160.0
mean       -2.8      95.6
std       116.4      93.5
min      -250.0     -51.0
25%       -53.0      14.0
50%         0.0      57.0
75%        45.0     183.0
max       250.0     734.0

LOC_X range : -250 → 250  (expected roughly −250 → 250)
LOC_Y range : -51 → 734  (expected roughly −50 → 750)


## Section 1.4 — Filter Out Invalid Coordinates

The NBA API data is generally clean, but a small number of rows can carry coordinate values that fall outside the physical dimensions of an NBA court. These usually represent encoding artifacts where a missing location was stored as a default value rather than left null.

We create a working copy called `cleaned` here and apply all remaining transformations to it. The original `raw` DataFrame stays untouched throughout the notebook so we always have something to compare against.

The bounds we use for filtering:

LOC_X must be between −260 and 260 raw units (−26 to 26 feet). The actual sidelines are at ±25 feet (±250 raw units). We use a slightly wider window so we do not accidentally clip any legitimate corner three attempts that the API may have logged with a coordinate just beyond the sideline.

LOC_Y must be between −80 and 900 raw units (−8 to 90 feet). The baseline behind the basket sits at roughly −4.75 feet (−47 raw units). We use −80 to give a small buffer. The upper bound of 900 (90 feet) covers everything from normal shots up to full-court heaves — any shot the API recorded as an attempt is kept here. The zone assignment step later will assign anything beyond half court (47 feet) to the Backcourt zone.

We print a summary showing how many rows were removed so we can confirm the filter is not too aggressive.

In [4]:
# Create the working copy — raw stays untouched from this point forward
cleaned = raw.copy()

n_before = len(cleaned)

# Filter: keep only rows within the physical bounds of an NBA court
coord_mask = (
    (cleaned["LOC_X"] >= -260) & (cleaned["LOC_X"] <= 260) &
    (cleaned["LOC_Y"] >= -80)  & (cleaned["LOC_Y"] <= 900)
)
cleaned = cleaned[coord_mask].copy()

n_after   = len(cleaned)
n_dropped = n_before - n_after

print(f"Rows before filter : {n_before:,}")
print(f"Rows after filter  : {n_after:,}")
print(f"Rows dropped       : {n_dropped:,}  ({n_dropped / n_before:.3%} of total)")

if n_dropped > 0:
    print("\nDropped row coordinate ranges:")
    dropped = raw[~coord_mask]
    print(f"  LOC_X : {dropped['LOC_X'].min()} → {dropped['LOC_X'].max()}")
    print(f"  LOC_Y : {dropped['LOC_Y'].min()} → {dropped['LOC_Y'].max()}")

Rows before filter : 219,160
Rows after filter  : 219,160
Rows dropped       : 0  (0.000% of total)


## Section 1.5 — Convert Coordinates to Feet

We divide LOC_X and LOC_Y by 10 to convert from the API's tenths-of-a-foot encoding to feet. The result goes into two new columns — LOC_X_FT and LOC_Y_FT — rather than overwriting the originals. Keeping both lets us cross-check if anything looks off later.

After conversion, known shot locations should land at predictable coordinates:

A left corner three should show LOC_X_FT near −22, LOC_Y_FT near 0 to 5.

A shot at the free-throw line should show LOC_X_FT near 0, LOC_Y_FT near 14 to 15.

A top-of-the-key three should show straight-line distance from (0, 0) of approximately 23.75 feet.

A dunk or layup in the restricted area should show distance under 4 feet.

The spot-check at the bottom of the code cell samples five rows from the data and shows the raw and converted values side by side so we can visually confirm the conversion is correct before moving on.

In [5]:
# Divide by 10 to convert tenths-of-a-foot to feet
cleaned["LOC_X_FT"] = cleaned["LOC_X"] / 10.0
cleaned["LOC_Y_FT"] = cleaned["LOC_Y"] / 10.0

# Compute straight-line distance from the basket for spot-check verification
cleaned["DIST_FT"] = np.sqrt(cleaned["LOC_X_FT"]**2 + cleaned["LOC_Y_FT"]**2).round(2)

# Spot-check: sample one row from each of four expected coordinate ranges
# to confirm the conversion matches known NBA court geometry
checks = pd.concat([
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "Restricted Area"].sample(1, random_state=1),
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "In The Paint (Non-RA)"].sample(1, random_state=1),
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "Left Corner 3"].sample(1, random_state=1),
    cleaned[cleaned["SHOT_ZONE_BASIC"] == "Above the Break 3"].sample(1, random_state=1),
])

spot_cols = ["NAME", "SHOT_ZONE_BASIC", "LOC_X", "LOC_Y", "LOC_X_FT", "LOC_Y_FT", "DIST_FT"]
print("Spot-check — one row per zone type:\n")
display(checks[spot_cols].reset_index(drop=True))

print(f"\nNew columns added to cleaned:")
print(f"  LOC_X_FT  range: {cleaned['LOC_X_FT'].min():.1f} → {cleaned['LOC_X_FT'].max():.1f} ft")
print(f"  LOC_Y_FT  range: {cleaned['LOC_Y_FT'].min():.1f} → {cleaned['LOC_Y_FT'].max():.1f} ft")
print(f"  DIST_FT   range: {cleaned['DIST_FT'].min():.1f} → {cleaned['DIST_FT'].max():.1f} ft")

Spot-check — one row per zone type:



,NAME,SHOT_ZONE_BASIC,LOC_X,LOC_Y,LOC_X_FT,LOC_Y_FT,DIST_FT
0,Tyrese Maxey,Restricted Area,10,-1,1.0,-0.1,1.00
1,Jaylen Brown,In The Paint (Non-RA),47,2,4.7,0.2,4.70
2,Donte DiVincenzo,Left Corner 3,-225,-5,-22.5,-0.5,22.51
3,Brandin Podziemski,Above the Break 3,-37,290,-3.7,29.0,29.24



New columns added to cleaned:
  LOC_X_FT  range: -25.0 → 25.0 ft
  LOC_Y_FT  range: -5.1 → 73.4 ft
  DIST_FT   range: 0.0 → 73.4 ft


## Section 1.6 — Before/After Summary

This is the acceptance check for the entire coordinate cleaning pipeline. We compare the raw ranges saved in Section 1.3 against the converted ranges in `cleaned` to confirm that the divide-by-10 conversion is exact, no rows were lost that should have been kept, and the cleaned values fall within expected NBA court dimensions.

If everything is correct, the cleaned foot values should be exactly one tenth of the raw tenths-of-a-foot values, the row count should match what we saw after the filter in Section 1.4, and the player count should match what Notebook 01 pulled.

This is the last cell in Section 1. Section 2 begins zone assignment.

In [6]:
print("=== Coordinate cleaning — before / after ===\n")

print(f"{'':30s}  {'RAW (tenths of ft)':>20s}  {'CLEANED (ft)':>15s}")
print(f"{'─'*70}")
print(f"{'LOC_X range':30s}  "
      f"{str(RAW_X_RANGE[0]) + ' → ' + str(RAW_X_RANGE[1]):>20s}  "
      f"{cleaned['LOC_X_FT'].min():.1f} → {cleaned['LOC_X_FT'].max():.1f}")
print(f"{'LOC_Y range':30s}  "
      f"{str(RAW_Y_RANGE[0]) + ' → ' + str(RAW_Y_RANGE[1]):>20s}  "
      f"{cleaned['LOC_Y_FT'].min():.1f} → {cleaned['LOC_Y_FT'].max():.1f}")

print(f"\n{'=== Dataset summary after Section 1 ==='}")
print(f"Total rows     : {len(cleaned):,}")
print(f"Unique players : {cleaned['NAME'].nunique():,}")
print(f"Columns        : {len(cleaned.columns)}  "
      f"(original {len(raw.columns)} + LOC_X_FT, LOC_Y_FT, DIST_FT)")

print(f"\nSection 1 complete. Proceeding to zone assignment.")

=== Coordinate cleaning — before / after ===

                                  RAW (tenths of ft)     CLEANED (ft)
──────────────────────────────────────────────────────────────────────
LOC_X range                               -250 → 250  -25.0 → 25.0
LOC_Y range                                -51 → 734  -5.1 → 73.4

=== Dataset summary after Section 1 ===
Total rows     : 219,160
Unique players : 582
Columns        : 28  (original 25 + LOC_X_FT, LOC_Y_FT, DIST_FT)

Section 1 complete. Proceeding to zone assignment.


## Section 2 — Zone Assignment and Base Metrics

Every shot in our dataset has an `(LOC_X_FT, LOC_Y_FT)` coordinate. Now we assign each one a named court zone, then compute shooting efficiency metrics per player per zone.

**A note  within ±26 ft, LOC_Y within −8 to 90 ft) are wider than any valid shot location, so no legitimate corner 3s or above-break 3s were removed. No changes needed.

**Section 2 steps:**
1. Define a zone assignment function using NBA court geometry
2. Apply it to every shot and verify no shots fall through as unassigned
3. Compute FG%, eFG%, PPS, and shot frequency per player per zone
4. Flag zones with fewer than 20 attempts as low-sample
5. Save the enriched DataFrame to `data/processed/shots_cleaned.csv`

### 2.1 — Zone Assignment Function

The NBA 3-point line is not a perfect arc. It has two straight segments near the corners (at exactly 22 feet from the basket) that connect to a circular arc (radius 23.75 feet) higher up the court. The point where the straight line meets the arc is the "break."

We can find the break using the Pythagorean theorem: if the straight corner line sits at |x| = 22 ft and the arc has radius 23.75 ft, the break is at y = √(23.75² − 22²) ≈ **8.95 ft** from the basket.

This means:
- A shot at `(−22.5, 4.0)` is a **left corner 3** — beyond 22 ft wide, below the break
- A shot at `(−22.5, 10.0)` is an **above-break 3 left** — same x, but above the break, so it falls on the arc and distance = √(22.5² + 10²) ≈ 24.6 ft ≥ 23.75 ft

The function works top-down: each shot is tested against zones in priority order, and the first match wins.

---
- A shot at `(−22.5, 4.0)` is a **left corner 3** — beyond 22 ft wide, below the break
- A shot at `(−22.5, 10.0)` is an **above-break 3 left** — same x, but above the break, so it falls on the arc and distance = √(22.5² + 10²) ≈ 24.6 ft ≥ 23.75 ft

The function works top-down: each shot is tested against zones in priority order, and the first match wins.

In [7]:
# Precompute once — the y-value where the arc (23.75 ft) meets the corner straight line (22 ft).
# Using Pythagorean theorem: y = sqrt(r_arc² - x_corner²)
CORNER_3_BREAK_Y = float(np.sqrt(23.75**2 - 22.0**2))   # ≈ 8.95 ft
print(f"Corner 3 / above-break boundary: y = {CORNER_3_BREAK_Y:.2f} ft from basket")

def assign_zone(loc_x_ft: float, loc_y_ft: float) -> str:
    """
    Assign a shot to one of 11 named court zones using NBA court geometry.

    Parameters
    ----------
    loc_x_ft : horizontal distance from basket center (negative = left, positive = right)
    loc_y_ft : depth from basket toward half court (0 = basket, ~47 = half court)

    Returns
    -------
    Zone name as a string — never None, so every shot gets a zone.
    """
    dist = np.sqrt(loc_x_ft**2 + loc_y_ft**2)   # straight-line distance from basket

    # --- Zone 1: Backcourt ---
    # Half court is ~47 ft from the basket. These are desperation heaves, not real shot attempts.
    if loc_y_ft > 47.0:
        return "Backcourt"

    # --- Zones 2 & 3: Corner 3s ---
    # The corner 3 straight line sits at |x| = 22 ft and extends from the baseline
    # up to the break point (y ≈ 8.95 ft). Any shot beyond 22 ft wide AND below the
    # break is a corner 3, regardless of its exact distance from the basket.
    if loc_x_ft <= -22.0 and loc_y_ft <= CORNER_3_BREAK_Y:
        return "Left Corner 3"
    if loc_x_ft >= 22.0 and loc_y_ft <= CORNER_3_BREAK_Y:
        return "Right Corner 3"

    # --- Zones 4, 5, 6: Above-break 3s ---
    # The arc portion of the 3PT line is a circle of radius 23.75 ft from the basket.
    # Any shot at or beyond 23.75 ft that isn't in a corner zone is above-break.
    if dist >= 23.75:
        if loc_x_ft < -7.0:
            return "Above Break 3 Left"
        elif loc_x_ft <= 7.0:
            return "Above Break 3 Center"
        else:
            return "Above Break 3 Right"

    # --- Zone 7: Restricted Area ---
    # A 4-ft radius semicircle around the basket. Shots here are almost always
    # at-rim attempts (layups, dunks, putbacks).
    if dist <= 4.0:
        return "Restricted Area"

    # --- Zone 8: Paint (Non-RA) ---
    # The painted rectangle is 16 ft wide (±8 ft from center) and extends from the
    # baseline to the free throw line (~14.25 ft from the basket, using 19 ft from
    # baseline minus the 4.75 ft basket-to-baseline distance). Shots 
    # baseline to the free throw line (~14.25 ft from the basket, using 19 ft from
    # baseline minus the 4.75 ft basket-to-baseline distance). Shots here outside
    # the restricted area are short mid-post attempts, floaters, and hook shots.
    if abs(loc_x_ft) <= 8.0 and loc_y_ft <= 14.5:
        return "Paint (Non-RA)"

    # --- Zones 9, 10, 11: Mid-range ---
    # Everything inside the 3PT line that isn't paint. Split left/center/right
    # using the same ±7 ft boundary as the above-break 3 zones for visual consistency.
    if loc_x_ft < -7.0:
        return "Mid-Range Left"
    elif loc_x_ft <= 7.0:
        return "Mid-Range Center"
    else:
        return "Mid-Range Right"

Corner 3 / above-break boundary: y = 8.95 ft from basket


### 2.2 — Apply Zone Assignment

We apply `assign_zone()` to every row in the DataFrame, then immediately check for any shots that didn't receive a valid zone. Because our function always returns a string and covers every possible coordinate, there should be zero nulls.

We also add two helper columns here that we'll need for metrics:
- `IS_3PT` — 1 if the shot was a 3-point attempt, 0 otherwise (read from the API's `SHOT_TYPE` column, which is the authoritative source)
- `POINTS` — how many points the shot was worth (2 or 3 if made, 0 if missed)

In [8]:
# Apply zone assignment — each row calls assign_zone() with its foot coordinates
cleaned["ZONE"] = cleaned.apply(
    lambda row: assign_zone(row["LOC_X_FT"], row["LOC_Y_FT"]),
    axis=1
)

# Add helper columns for metric computation
cleaned["IS_3PT"] = cleaned["SHOT_TYPE"].str.startswith("3PT").astype(int)
cleaned["POINTS"]  = cleaned["SHOT_MADE_FLAG"] * (2 + cleaned["IS_3PT"])

# Verify: every shot must have a zone
n_unassigned = cleaned["ZONE"].isna().sum()
print(f"Unassigned shots: {n_unassigned}")  # must be 0

print(f"\nZones in use ({cleaned['ZONE'].nunique()}):")
for zone in sorted(cleaned["ZONE"].unique()):
    print(f"  {zone}")

Unassigned shots: 0

Zones in use (11):
  Above Break 3 Center
  Above Break 3 Left
  Above Break 3 Right
  Backcourt
  Left Corner 3
  Mid-Range Center
  Mid-Range Left
  Mid-Range Right
  Paint (Non-RA)
  Restricted Area
  Right Corner 3


### 2.3 — Verify: Shot Count Per Player Per Zone

Before computing any efficiency metrics, we confirm that every player has shots assigned across multiple zones and that no zone is suspiciously empty. A pivot table (players as rows, zones as columns) makes this easy to scan.

Any `0` in this table is worth noting — it might mean a player genuinely never took a shot from that zone, or it could signal a zone boundary that's miscutting real attempts.
Any `0` in this table is worth noting — it might mean a player genuinely never took a shot from that zone, or it could signal a zone boundary that's miscutting real attempts.

In [9]:
zone_order = [
    "Restricted Area", "Paint (Non-RA)",
    "Mid-Range Left", "Mid-Range Center", "Mid-Range Right",
    "Left Corner 3", "Right Corner 3",
    "Above Break 3 Left", "Above Break 3 Center", "Above Break 3 Right",
    "Backcourt",
]

shot_counts = (
    cleaned
    .groupby(["NAME", "ZONE"])
    .size()
    .unstack(fill_value=0)       # zones become columns
    .reindex(columns=zone_order, fill_value=0)   # consistent column order
)

print("Shot counts per player per zone:\n")
display(shot_counts)

# Flag any zone where a player has 0 attempts
zeros = [(player, zone) for player in shot_counts.index
         for zone in shot_counts.columns if shot_counts.loc[player, zone] == 0]
if zeros:
    print(f"\nZones with 0 attempts:")
    for player, zone in zeros:
        print(f"  {player:<20} — {zone}")
else:
    print("\nNo empty zones detected.")

Shot counts per player per zone:



ZONE,Restricted Area,Paint (Non-RA),Mid-Range Left,Mid-Range Center,Mid-Range Right,Left Corner 3,Right Corner 3,Above Break 3 Left,Above Break 3 Center,Above Break 3 Right,Backcourt
NAME,,,,,,,,,,,
A.J. Lawson,27,6,0,0,0,21,11,6,2,5,0
AJ Green,19,19,10,2,14,81,72,156,84,161,0
AJ Johnson,77,38,2,0,2,4,7,9,13,24,0
Aaron Gordon,129,65,17,7,23,25,21,46,14,51,0
Aaron Holiday,22,44,6,4,6,32,22,54,18,44,0
...,...,...,...,...,...,...,...,...,...,...,...
Zach LaVine,120,78,36,22,37,34,20,96,37,67,0
Zeke Nnaji,75,12,5,2,3,8,10,9,18,9,0
Ziaire Williams,114,53,9,1,5,45,49,49,21,87,0



Zones with 0 attempts:
  A.J. Lawson          — Mid-Range Left
  A.J. Lawson          — Mid-Range Center
  A.J. Lawson          — Mid-Range Right
  A.J. Lawson          — Backcourt
  AJ Green             — Backcourt
  AJ Johnson           — Mid-Range Center
  AJ Johnson           — Backcourt
  Aaron Gordon         — Backcourt
  Aaron Holiday        — Backcourt
  Aaron Nesmith        — Backcourt
  Aaron Wiggins        — Backcourt
  Ace Bailey           — Backcourt
  Adama Bal            — Mid-Range Center
  Adama Bal            — Backcourt
  Adem Bona            — Mid-Range Center
  Adem Bona            — Mid-Range Right
  Adem Bona            — Right Corner 3
  Adem Bona            — Above Break 3 Left
  Adem Bona            — Above Break 3 Center
  Adem Bona            — Backcourt
  Adou Thiero          — Mid-Range Left
  Adou Thiero          — Mid-Range Center
  Adou Thiero          — Mid-Range Right
  Adou Thiero          — Left Corner 3
  Adou Thiero          — Right Corner 3
  Ad

### 2.4 — Base Metrics Per Player Per Zone

Now we compute four shooting metrics for each player-zone combination:

| Metric | Formula | What it tells you |
|---|---|---|
| **FG%** | FGM / FGA | Raw make rate — ignores shot value |
| **eFG%** | (FGM + 0.5 × 3PM) / FGA | Adjusts for the extra point on 3s — the key efficiency metric |
| **PPS** | Total points / FGA | Average points generated per attempt — most directly comparable across zones |
| **Shot frequency** | Zone FGA / Player's total FGA | How often this player chooses this zone |

eFG% is more meaningful than FG% because a player who makes 40% of their 3s (eFG% = 0.60) is actually more efficient than one who makes 50% of their 2s (eFG% = 0.50), since three-pointers generate more value per attempt.

---value per attempt.

In [10]:
def compute_zone_metrics(group: pd.DataFrame) -> pd.Series:
    """Compute shooting efficiency metrics for one player-zone group."""
    fga  = len(group)
    fgm  = group["SHOT_MADE_FLAG"].sum()
    fg3m = (group["SHOT_MADE_FLAG"] * group["IS_3PT"]).sum()
    pts  = group["POINTS"].sum()

    return pd.Series({
        "FGA"      : fga,
        "FGM"      : fgm,
        "3PM"      : fg3m,
        "FG_PCT"   : round(fgm / fga, 4)                    if fga > 0 else np.nan,
        "EFG_PCT"  : round((fgm + 0.5 * fg3m) / fga, 4)    if fga > 0 else np.nan,
        "PPS"      : round(pts / fga, 4)                    if fga > 0 else np.nan,
    })

# Group by player + zone and apply metrics
metrics = (
    cleaned
    .groupby(["NAME", "ZONE"])
    .apply(compute_zone_metrics)
    .reset_index()
)

# Shot frequency: each zone's FGA as a share of that player's total FGA
metrics["SHOT_FREQ"] = (
    metrics.groupby("NAME")["FGA"]
    .transform(lambda x: x / x.sum())
    .round(4)
)

print(f"Metrics table shape: {metrics.shape}")
print(f"\nSample rows (LeBron James):\n")
display(
    metrics[metrics["NAME"] == "LeBron James"]
    .set_index("ZONE")
    .reindex(zone_order)
    [["FGA", "FG_PCT", "EFG_PCT", "PPS", "SHOT_FREQ"]]
)

Metrics table shape: (5008, 9)

Sample rows (LeBron James):



,FGA,FG_PCT,EFG_PCT,PPS,SHOT_FREQ
ZONE,,,,,
Restricted Area,341.0,0.7566,0.7566,1.5132,0.3711
Paint (Non-RA),180.0,0.4167,0.4167,0.8333,0.1959
Mid-Range Left,83.0,0.3976,0.3976,0.7952,0.0903
Mid-Range Center,29.0,0.4828,0.4828,0.9655,0.0316
Mid-Range Right,43.0,0.3721,0.3721,0.7442,0.0468
Left Corner 3,27.0,0.2222,0.3333,0.6667,0.0294
Right Corner 3,22.0,0.3636,0.5455,1.0909,0.0239
Above Break 3 Left,97.0,0.3608,0.5412,1.0825,0.1055
Above Break 3 Center,38.0,0.2632,0.3947,0.7895,0.0413


### 2.5 — Low-Sample Flag

Any zone where a player took fewer than 20 attempts produces unreliable efficiency numbers. A sample of 10 shots could show 60% eFG% purely by luck. We don't throw these rows away — small samples are still informative context — but we flag them so analysis notebooks can filter or label them appropriately.

The threshold of 20 attempts is a common minimum in zone-level NBA analytics (roughly the point where eFG% stabilizes within ±10 percentage points).are still informative context — but we flag them so analysis notebooks can filter or label them appropriately.

The threshold of 20 attempts is a common minimum in zone-level NBA analytics (roughly the point where eFG% stabilizes within ±10 percentage points).

In [11]:
MIN_ATTEMPTS = 20

metrics["LOW_SAMPLE"] = metrics["FGA"] < MIN_ATTEMPTS

n_low  = metrics["LOW_SAMPLE"].sum()
n_total = len(metrics)
print(f"Low-sample zones (< {MIN_ATTEMPTS} attempts): {n_low} of {n_total} player-zone rows")
print()

# Show the flagged zones so we know what to treat carefully downstream
low_sample_rows = metrics[metrics["LOW_SAMPLE"]][["NAME", "ZONE", "FGA"]].sort_values("FGA")
display(low_sample_rows.reset_index(drop=True))

Low-sample zones (< 20 attempts): 2442 of 5008 player-zone rows



,NAME,ZONE,FGA
0,Noah Penda,Mid-Range Center,1.0
1,Stanley Umude,Paint (Non-RA),1.0
2,Moussa Diabaté,Mid-Range Center,1.0
3,Gui Santos,Mid-Range Center,1.0
4,Caleb Houstan,Paint (Non-RA),1.0
...,...,...,...
2437,Pete Nance,Above Break 3 Center,19.0
2438,Asa Newell,Paint (Non-RA),19.0
2439,Austin Reaves,Mid-Range Left,19.0
2440,Ousmane Dieng,Left Corner 3,19.0


### 2.6 — Save to `data/processed/`

We save the shot-level DataFrame (with zone labels, IS_3PT, and POINTS columns added) to `data/processed/shots_cleaned.csv`. This is the single input file for every notebook that follows — no notebook after this one ever reads from `data/raw/`.

We do not save the metrics table here; each analysis notebook will recompute its own aggregations from the shot-level data, which keeps things flexible.

In [12]:
output_path = PROCESSED_DIR / "shots_cleaned.csv"
cleaned.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows  : {len(cleaned):,}")
print(f"Cols  : {len(cleaned.columns)}")
print(f"\nColumns added in this notebook:")
new_cols = ["LOC_X_FT", "LOC_Y_FT", "ZONE", "IS_3PT", "POINTS"]
for col in new_cols:
    print(f"  {col:<15} dtype={cleaned[col].dtype}   sample={cleaned[col].iloc[0]}")

print(f"\nZone distribution across all players:")
print(
    cleaned["ZONE"]
    .value_counts()
    .reindex(zone_order)
    .to_string()
)

Saved: /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/shots_cleaned.csv
Rows  : 219,160
Cols  : 30

Columns added in this notebook:
  LOC_X_FT        dtype=float64   sample=-5.0
  LOC_Y_FT        dtype=float64   sample=27.4
  ZONE            dtype=str   sample=Above Break 3 Center
  IS_3PT          dtype=int64   sample=1
  POINTS          dtype=int64   sample=0

Zone distribution across all players:
ZONE
Restricted Area         62278
Paint (Non-RA)          44573
Mid-Range Left           8313
Mid-Range Center         4741
Mid-Range Right          8281
Left Corner 3           12308
Right Corner 3          11460
Above Break 3 Left      27636
Above Break 3 Center    14834
Above Break 3 Right     24721
Backcourt                  15
